In [ ]:
# Copyright (C) 2026 Advanced Micro Devices, Inc. All rights reserved.
#
# SPDX-License-Identifier: BSD-3-Clause
#
# Portions of this file consist of AI-generated content. AI-assisted
# content has been reviewed and validated by the authors.

# Language on the wire: the Llama intent node

Component 1 taught the brain to turn a sentence into
`{"command", "object"}`. This notebook makes that brain a ROS 2 citizen.

![Alt Text](../../animations/Intent_Node_Animation.gif)

This is a **text-based pipeline**: commands arrive as plain text on
`/vla/transcript`, published by a terminal, a test, a UI, or any other
program. Why a node? Decoupling. Whatever produces text just publishes it;
ANY node can react to intents (the orchestrator today, a logger tomorrow, a
second robot next week) and none of them link against the LLM. You can
inject language from a terminal, which is priceless for debugging:
`ros2 topic pub /vla/transcript ...` and the robot "hears".

Two design rules carry over and are the heart of your TODOs:

1. **The stop fast path.** `is_stop()` keyword-matches the RAW transcript
   and publishes immediately. The safety stop never waits for a language
   model.
2. **The noise gate.** Upstream text sources deliver junk sometimes:
   empty strings, boilerplate like "Thank you.", fragments. With a grammar
   that *guarantees* a valid command out, a junk transcript would become a
   real robot behavior. `is_noise()` drops it before the LLM.

Also note the QoS flip: transcripts and intents use **RELIABLE** delivery
(the rclpy default). A lost camera frame is free; a lost "stop" is not.

**Matching TODOs:** `2.10`-`2.11` in `project/ros/intent_node.py`.

In [ ]:
# workshop bootstrap: run against the finished reference code in solution/
# (no TODO stubs), so every cell works out of the box. To exercise YOUR
# implementations instead, change "solution" to "project" and restart.
import sys, pathlib
import threading, time
import json

PROJECT = (pathlib.Path.cwd() / ".." / ".." / "solution").resolve()
assert (
    PROJECT / "config" / "workshop.yaml"
).exists(), f"solution not found at {PROJECT}"
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

from common.quiet import hush_benign_warnings

hush_benign_warnings()  # silence known-benign third-party log noise

from common.config import load_config, resolve, PROJECT_ROOT
import ipywidgets as widgets
from IPython.display import display
from jupyter_ui_poll import ui_events
import requests
from common.intents import Intent, ParsedCommand, is_noise, is_stop
from common.fixtures import offline_parse_text
from common.intents import INTENT_SYSTEM_PROMPT

cfg = load_config()
print("running against:", PROJECT_ROOT)

In [ ]:
# ROS 2 check: this notebook runs on real ROS 2 (rclpy) only.
try:
    import rclpy

    HAVE_ROS = True
except ImportError:
    HAVE_ROS = False

ROS_HELP = """rclpy is not importable in this kernel, so the ROS 2 cells cannot run."""


def require_ros():
    if not HAVE_ROS:
        raise RuntimeError(ROS_HELP)


if HAVE_ROS:
    if not rclpy.ok():
        rclpy.init()
    print("real ROS 2 (rclpy) ready")
else:
    print(ROS_HELP)

> **About background nodes.** The demo nodes this notebook creates are
> self-contained: they publish their own data, so the lesson works with
> nothing else running. To also run the *real* device nodes (the SO-101
> arm, the cameras, the intent node), start them yourself in a terminal;
> the "On the rig" section below has the exact commands. Every subscriber
> simply reads whatever is on the topic, so both can coexist.

In [ ]:
require_ros()

# A tiny helper to spin nodes on background executors, so the notebook
# stays interactive while nodes tick. (This mirrors how the pipeline's
# orchestrator embeds its client nodes.)


class Runner:
    def __init__(self, node):
        from rclpy.executors import SingleThreadedExecutor

        self.node, self.executor = node, SingleThreadedExecutor()
        self.executor.add_node(node)
        threading.Thread(target=self.executor.spin, daemon=True).start()

    def stop(self):
        self.executor.shutdown()
        try:
            self.node.destroy_node()
        except Exception:
            pass


def wait_for(pred, timeout_s=3.0, interval=0.02):
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        v = pred()
        if v:
            return v
        time.sleep(interval)
    return None

## 1. A minimal intent node

The parser is **injected** (`str -> ParsedCommand`), and it is built
exactly the way the pipeline builds it: the real `LlamaIntentParser` -
Llama 3.2 3B on the iGPU, GBNF grammar attached - whenever the
llama-server is reachable (or can be autostarted per the config), with the
keyword stand-in only as a fallback so the lesson runs on any machine.
The difference matters: free language like *"bust a move"* only routes to
`dance` through the real brain; the keyword stand-in knows only the
literal verbs. The node doesn't know or care which it got - that seam is
what the harness exploits to test your wiring without a GPU. We wrap the
parser to *count calls*, because the whole point of section 3 is proving
certain messages never reach it.


In [ ]:
require_ros()

from rclpy.node import Node
from std_msgs.msg import String

# Build the parser the way the pipeline does: the real Llama brain when
# the server is reachable (or autostarts per config), else the keyword
# stand-in - the same cascade app.py and the real intent node use.
llama_parser = None


def make_parser():
    global llama_parser
    try:
        from models.llm import LlamaIntentParser

        llama_parser = LlamaIntentParser(cfg)  # connect or autostart (config)
        print("parser: Llama 3.2 3B on the iGPU (grammar-constrained)")
        return llama_parser.parse
    except Exception as e:
        print(f"parser: llama unavailable ({e}) - keyword stand-in")

        def offline(text):
            d = offline_parse_text(text)
            params = {"object": d["object"]} if d.get("object") else {}
            return ParsedCommand(
                Intent.from_string(d["command"]), params=params, transcript=text
            )

        return offline


_parse = make_parser()

parser_calls = []


def demo_parser(text):
    parser_calls.append(text)  # count every LLM "round trip"
    return _parse(text)


class DemoIntentNode(Node):
    def __init__(self, parser):
        super().__init__("demo_intent_node")
        self._parse = parser
        # RELIABLE (default QoS, depth 10): intents are commands, not sensor data.
        self.create_subscription(String, "/vla/transcript", self.on_transcript, 10)
        self.pub = self.create_publisher(String, "/vla/intent", 10)

    def on_transcript(self, msg):
        text = msg.data or ""
        if is_stop(text):  # 1) safety first, LLM never
            self.emit(ParsedCommand(Intent.STOP, transcript=text))
            return
        if is_noise(text):  # 2) junk gate
            return
        self.emit(self._parse(text))  # 3) the brain

    def emit(self, cmd):
        self.pub.publish(
            String(
                data=json.dumps(
                    {"command": cmd.intent.value, "object": cmd.object_name or ""}
                )
            )
        )


intent_node = DemoIntentNode(demo_parser)
node_runner = Runner(intent_node)
print("intent node up:", intent_node.get_name())

## 2. Drive it from the client side

In [ ]:
require_ros()

received = []
probe = rclpy.create_node("notebook_probe")
probe.create_subscription(
    String, "/vla/intent", lambda m: received.append(json.loads(m.data)), 10
)
transcript_pub = probe.create_publisher(String, "/vla/transcript", 10)
probe_runner = Runner(probe)
time.sleep(0.3)  # let DDS discovery settle


def say(text, wait=1.0):
    n = len(received)
    transcript_pub.publish(String(data=text))
    wait_for(lambda: len(received) > n, wait)
    got = received[-1] if len(received) > n else None
    print(f"  {text!r:<38} -> {got}")
    return got


print("transcript -> intent over the wire:")
say("pick up the ball")
say("can you copy my movements")
say("show me your best dance moves")
say("bust a move", wait=3.0)  # free language: only the real brain gets this
say("go home please")

## 3. Prove the two gates work

The counting parser is our witness: a **stop** must produce an intent
*without* a parser call, and **noise** must produce neither.

In [ ]:
require_ros()

calls_before, intents_before = len(parser_calls), len(received)

got = say("whoa whoa STOP right now")
assert got and got["command"] == "stop", "stop must be published"
assert len(parser_calls) == calls_before, "stop must NOT invoke the parser!"
print(
    f"  OK: stop routed in ~0 ms with ZERO parser calls "
    f"(parser count still {len(parser_calls)})"
)

say("Thank you.", wait=0.6)  # classic junk text the noise gate must eat
assert len(received) == intents_before + 1, "noise must publish NO intent"
assert len(parser_calls) == calls_before, "noise must NOT invoke the parser"
print("  OK: noise dropped - no intent, no parser call")

## 4. Talk to the node, live

The suite above proved the gates. Now drive it yourself: whatever you type
below is published on `/vla/transcript`, crosses the wire, runs the
stop -> noise -> parse gauntlet inside the node, and whatever lands on
`/vla/intent` is printed next to it - along with which path it took (the
parser-call counter is our witness again). Try `pick up the ball`, a bare
`STOP`, junk like `Thank you.`, and something it has no verb for. Press
**done** to release the cell.


In [ ]:
require_ros()

box = widgets.Text(
    placeholder="say something, e.g. pick up the ball",
    continuous_update=False,  # sync the value once, on Enter/blur
    layout=widgets.Layout(width="420px"),
)
send_btn = widgets.Button(description="send", button_style="primary")
done_btn = widgets.Button(description="done", button_style="danger")
console = widgets.Output(layout=widgets.Layout(border="1px solid #444", padding="6px"))
finished = {"flag": False}
done_btn.on_click(lambda _: finished.update(flag=True))


def send(_=None):
    text = box.value.strip()
    if not text:
        return
    box.value = ""
    with console:
        n, calls = len(received), len(parser_calls)
        transcript_pub.publish(String(data=text))
        wait_for(lambda: len(received) > n, 1.5)
        got = received[-1] if len(received) > n else None
        if len(parser_calls) > calls:
            via = "LLM parse"
        elif got and got.get("command") == "stop":
            via = "stop fast path - parser never called"
        else:
            via = "noise gate - dropped, no parser call"
        print(f"you : {text!r}")
        print(f"node: {got}   [{via}]\n")


send_btn.on_click(send)
# Enter (or clicking away) commits the text; the observer sends it.
box.observe(send, "value")
display(widgets.VBox([widgets.HBox([box, send_btn, done_btn]), console]))

with ui_events() as poll:
    while not finished["flag"]:
        poll(10)  # keep send/done clicks flowing
        time.sleep(0.05)
print("console closed")

Why so strict about latency? The stop keyword path skips the LLM round
trip *and* every failure mode of a busy server: a queued slot, a slow
generation, a hung connection. Worst case matters more than average case
when the arm is mid-swing.

## 5. The real brain: grammar-constrained parsing (TODO 1.6)

Section 1's node already got the real parser when the server was up.
Under the hood it is one HTTP call to the llama-server on the iGPU
(`01_ai_models/03_igpu_llama.ipynb` brought it up) - with a **GBNF
grammar** attached, so the reply *cannot* be malformed. llama.cpp does *constrained decoding*: at every step, tokens
that would violate the grammar are masked out, and the model physically
cannot emit anything but

```
{"command": "<one of nine known commands>", "object": "<0-2 lowercase words>"}
```

Parsing failures aren't handled. They're *impossible by construction*.
Two details worth noticing: unknown verbs collapse into `"unknown"` (the
grammar *constrains*, the system prompt *steers* - `INTENT_SYSTEM_PROMPT`
in `common/intents.py`), and the `object` slot is how *"grab the
**bottle**"* carries its noun to the behavior that needs it.

This is TODO 1.6 (`models/llm.py`), and swapping it in for the keyword
stand-in is one constructor argument: the node neither knows nor cares.


In [ ]:
l = cfg["llm"]
BASE = f"http://{l['host']}:{l['port']}"
try:
    SERVER_UP = requests.get(f"{BASE}/health", timeout=0.5).status_code == 200
except requests.RequestException:
    SERVER_UP = False

if not SERVER_UP:
    print(f"(no llama-server at {BASE} - the keyword stand-in above keeps the")
    print(" node fully functional. Start the server via ./run_notebooks.sh or")
    print(" the recipe in 01_ai_models/03_igpu_llama.ipynb, then re-run.)")
else:
    GRAMMAR = resolve(l["grammar_file"]).read_text()
    http = requests.Session()  # keep-alive: saves ms per command

    def parse_intent(transcript):
        payload = {
            "messages": [
                {"role": "system", "content": INTENT_SYSTEM_PROMPT},
                {"role": "user", "content": transcript.strip()},
            ],
            "temperature": 0.0,  # routing must be deterministic
            "max_tokens": 48,  # the grammar output is tiny
            "grammar": GRAMMAR,  # <- THE line that makes it robot-grade
        }
        r = http.post(f"{BASE}/v1/chat/completions", json=payload, timeout=30)
        r.raise_for_status()
        return json.loads(r.json()["choices"][0]["message"]["content"])

    for text in (
        "pick up the ball",
        "can you copy my movements",
        "show me your best dance moves",
        "hold this pencil for me",
        "what's the weather like today",
    ):
        t0 = time.perf_counter()
        d = parse_intent(text)
        print(f"  {text!r:<38} -> {d}   ({1e3*(time.perf_counter()-t0):.0f} ms)")

Expect roughly 100 ms per command: the GBNF output is a dozen tokens, and
with the keep-alive session most of the budget is generation.

## 6. On the rig

After your TODOs (and with component 1's `LlamaIntentParser` implemented):

```bash
# Terminal 1 - real brain (or --mock for the keyword stand-in):
python -m ros.intent_node [--mock]

# Terminal 2 - inject language, watch intents:
ros2 topic pub --once /vla/transcript std_msgs/msg/String "{data: pick up the ball}"
ros2 topic echo /vla/intent
```

Anything can publish onto the transcript topic: a human at a terminal, a
test harness, the live UI. Your node neither knows nor cares who is
talking.


In [ ]:
# housekeeping
probe_runner.stop()
node_runner.stop()
if llama_parser is not None:
    llama_parser.close()  # stops a llama-server WE spawned; an externally
    # started one (run_notebooks.sh) keeps running
print("demo nodes stopped")

## 7. Your turn

Open **`project/ros/intent_node.py`**:

| TODO | What | Taught in |
|---|---|---|
| **2.10** | `IntentNode._setup_ros`: String sub/pub, RELIABLE | section 1 |
| **2.11** | `IntentNode._on_transcript`: stop -> noise -> parse -> publish | sections 1 and 3 |

Then, from the `project/` directory:

```bash
python -m ros.selftest

# to see all the TODOs you have left
python app.py
```

The harness publishes transcripts at your node and *counts parser calls*.
A stop that consults the LLM, or noise that slips through, gets named
feedback, not a red X.

That's component 2. Next: `03_integration/01_behavior.ipynb`, where models
and topics become a robot that mirrors you.